# Sprints Evaluator–Optimizer Loop — Offline Evidence Notebook

**Live integration:** Groq returns typed Pydantic feedback and a separate model call judges it. This notebook uses the supplied CSV for deterministic offline outputs and a local simulation of the retry rule. See `README.md` for the live Groq, Chroma, and Docker workflow.

## 1. Inspect the actual starter CSV

In [1]:
import csv
from pathlib import Path

data_path = Path("data/sprints_learner_db.csv")
if not data_path.is_file():
    data_path = Path("../data/sprints_learner_db.csv")
with data_path.open(encoding="utf-8-sig", newline="") as csv_file:
    rows = list(csv.DictReader(csv_file))
print("Actual rows:", len(rows))
print("Columns:", ", ".join(rows[0]))

Actual rows: 25
Columns: learner_id, name, bootcamp_track, completed_modules, python_assignment_score, target_salary_egp, preferred_employers, readiness_status


## 2. Evidence-based learner record — no hallucinated skills

In [2]:
student = next(r for r in rows if r["learner_id"] == "SPR-1001")
print(student["learner_id"], "| modules:", student["completed_modules"], "| score:", student["python_assignment_score"], "| salary EGP:", student["target_salary_egp"])

SPR-1001 | modules: 12 | score: 95 | salary EGP: 30000


## 3. Deterministic mock grading (not sandbox execution)

In [3]:
checks = [5 == 5, -3 + 1 == -2, 0 + 0 == 0]
observed_score = round(100 * sum(checks) / len(checks))
print("Mock tests:", sum(checks), "/", len(checks))
print("Expected grade:", observed_score, "pass:", observed_score >= 70)

Mock tests: 3 / 3
Expected grade: 100 pass: True


## 4. Evaluator–Optimizer attempts stop at three

In [4]:
def dry_run_optimizer(judge_approvals):
    """Trace the Sprints Assessor's three-attempt rubric review rule."""
    count = 0
    for approved in judge_approvals:
        count += 1
        print(f"Judge attempt {count}: {'approved' if approved else 'rejected'}")
        if approved or count == 3:
            break
    print("Total attempts:", count)

dry_run_optimizer([False, False, False, True])

Judge attempt 1: rejected
Judge attempt 2: rejected
Judge attempt 3: rejected
Total attempts: 3


## 5. Human approval for tier-1 examples

In [5]:
tier1 = {"opening_id": "DEMO-JOB-1", "employer": "Fawry", "tier_1": True, "source_status": "DEMO_NOT_VERIFIED"}
print("Needs LangGraph interrupt:", bool(tier1["tier_1"]))
print("Opening verified:", tier1["source_status"] == "VERIFIED_PARTNER_RECORD")

Needs LangGraph interrupt: True
Opening verified: False


## 6. Live workflow after configuring Groq and Gemini embeddings

In [6]:
print("Live workflow: python -m src.main init; python -m src.main ingest; docker build -f infra/Dockerfile.sandbox -t sprints-python-sandbox:local .; python -m src.main shell --learner SPR-1001")

Live workflow: python -m src.main init; python -m src.main ingest; docker build -f infra/Dockerfile.sandbox -t sprints-python-sandbox:local .; python -m src.main shell --learner SPR-1001
